## Libraries

In [1]:
#General Purpose
import copy
import os
from tqdm import tqdm

#GUI
import ipywidgets as widgets
from ipywidgets import interact

#Maths and Plotting
import numpy as np
import scipy
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1 import make_axes_locatable
import plotly.express as px
import seaborn as sns

#Stochastic Methods
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

#Machine Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import Dataset, TensorDataset, DataLoader

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: {}".format(DEVICE))

Using device: cpu


## Dataset Dictionaries

In [3]:
name_to_ID = {
    'Hard_Yellow_Cylinder_Small':0,
    'Hard_Orange_Cylinder_Small':1,
    'Hard_Red_Cylinder':2,
    'Hard_Violet_Cylinder':3,
    'Hard_Green_Cylinder':4,
    'Hard_Yellow_Cylinder_Big':5,
    'Hard_Orange_Cylinder_Big':6,
    'Hard_Blue_Cylinder':7,
    'Hard_Rubiks_Cube':8,
    'Hard_Spam':9,
    'Soft_Blue_Cube':10,
    'Soft_Blue_Die_Small':11,
    'Soft_Blue_Die_Big':12,
    'Soft_Brown_Cuboid':13,
    'Soft_Kinova_Cube':14,
    'Soft_Rosa_Cylinder':15,
    'Soft_Rosa_Die':16,
    'Soft_White_Die':17,
    'Soft_Yellow_Cube':18,
    'Soft_Yellow_Sponge':19,
    'Without_Object':20
}

name_to_color = {
    'Hard_Yellow_Cylinder_Small':'yellow',
    'Hard_Orange_Cylinder_Small':'orange',
    'Hard_Red_Cylinder':'red',
    'Hard_Violet_Cylinder':'purple',
    'Hard_Green_Cylinder':'green',
    'Hard_Yellow_Cylinder_Big':'darkgoldenrod',
    'Hard_Orange_Cylinder_Big':'sienna',
    'Hard_Blue_Cylinder':'blue',
    'Hard_Rubiks_Cube':'lightgreen',
    'Hard_Spam':'navy',
    'Soft_Blue_Cube':'cyan',
    'Soft_Blue_Die_Small':'gray',
    'Soft_Blue_Die_Big':'aquamarine',
    'Soft_Brown_Cuboid':'maroon',
    'Soft_Kinova_Cube':'tan',
    'Soft_Rosa_Cylinder':'deeppink',
    'Soft_Rosa_Die':'palevioletred',
    'Soft_White_Die':'lightsteelblue',
    'Soft_Yellow_Cube':'olive',
    'Soft_Yellow_Sponge':'gold',
    'Without_Object':'black'
}

load_to_exp = {
    "45" : 400,
    "120": 200,
    "150": 0
}

load_to_marker_matplotlib = {
    "45" : 'o',
    "120": 'x',
    "150": '^'
}

load_to_marker_plotly = {
    "45" : 'circle',
    "120": 'square',
    "150": 'diamond'
} # Available 2D Markers: circle, square, diamond, cross, x, triangle-up, triangle-down, triangle-left, triangle-right, pentagon, hexagon, star, hourglass, bowtie
# Available 3D Markers: circle, square, diamond, cross, x, circle-open, square-open, diamond-open, cross-open, x-open

signals_list = ["Time","Fingert._ID2","Bottom_ID2","Fingert._ID4","Middle_ID4","Bottom_ID4",
                "Fingert._ID2_comp","Bottom_ID2_comp","Fingert. ID4 comp","Middle ID4 comp","Bottom ID4 comp",
                "Motor ID2 Current","Motor ID4 Current"]

## File read

In [4]:
all_data_df = pd.read_pickle("./all_data_df.pkl")

## Loading all the data

In [5]:

analyse_signal_checkboxes = [widgets.Checkbox(value=False, description=signal_name) for signal_name in signals_list]
analyse_signal_column = widgets.VBox([
    widgets.HTML("<center><b>List of Signals</b></center>"),
    *analyse_signal_checkboxes
])

analyse_series_checkboxes = [widgets.Checkbox(value=False, description=load) for load in sorted(all_data_df['Series'].unique())]
analyse_series_column = widgets.VBox([
    widgets.HTML("<center><b>List of Data</b></center>"),
    *analyse_series_checkboxes
])

analyse_object_checkboxes = [widgets.Checkbox(value=False, description=object_name) for object_name in sorted(all_data_df['Object_Name'].unique())]
analyse_object_column = widgets.VBox([
    widgets.HTML("<center><b>List of Objects</b></center>"),
    *analyse_object_checkboxes
])

row = widgets.HBox([analyse_signal_column, analyse_series_column, analyse_object_column])
display(row)

In [6]:
analyse_signals_list = [checkbox.description for checkbox in analyse_signal_checkboxes if checkbox.value==True]
analyse_series_list = [checkbox.description for checkbox in analyse_series_checkboxes if checkbox.value==True]
analyse_objects_list = [checkbox.description for checkbox in analyse_object_checkboxes if checkbox.value==True]

## LSTM

In [7]:
# Data selection
lstm_df = all_data_df[(all_data_df["Object_Name"].isin(analyse_objects_list))]
lstm_df = lstm_df[(lstm_df["Series"].isin(analyse_series_list))]

recordings = list()

for _, row in tqdm(lstm_df.iterrows()):
        recordings.append(np.column_stack([row[signal_name] for signal_name in analyse_signals_list]))

#downsample_factor = 10
#ds_recordings = [rec[::downsample_factor,:] for rec in recordings]
#recordings = ds_recordings

labels = lstm_df["Object_ID"].to_numpy()
Num_IDs = lstm_df["Object_ID"].nunique()

le = LabelEncoder()
labels_encoded = le.fit_transform(labels)   # converts any labels → 0, 1, 2, ...
labels = labels_encoded

# Human-readable names for the encoded classes (used by classification_report
# and the confusion-matrix plot instead of bare integer indices).
ID_to_name = {v: k for k, v in name_to_ID.items()}
class_names = [ID_to_name.get(int(c), "ID_{}".format(int(c))) for c in le.classes_]

print("Recordings: {}; Labels: {}, Objects: {}".format(len(recordings), len(labels), Num_IDs))

lengths = np.array([rec.shape[0] for rec in recordings], dtype=np.int64)
num_channels = len(analyse_signals_list)

# NOTE: no globally padded tensor is built here.
# TimeSeriesDataset keeps the raw variable-length recordings and collate_fn
# pads each batch to its own maximum, so a dense (N, max_len, C) array would be
# allocated for nothing. Only the lengths are needed up front.
print("Channels: {}".format(num_channels))
print("Sequence lengths: min={}, max={}".format(lengths.min(), lengths.max()))

1532it [00:00, 2294.38it/s]

Recordings: 1532; Labels: 1532, Objects: 9
Channels: 7
Sequence lengths: min=5000, max=15000


In [8]:
class TimeSeriesDataset(Dataset):
    """Stores raw (unpadded) recordings. Padding happens per batch."""

    def __init__(self, recordings: list, lengths: np.ndarray, labels: np.ndarray):
        self.recordings = recordings   # list of (T_i, 7) arrays — no padding
        self.lengths    = torch.tensor(lengths, dtype=torch.int64)
        self.labels     = torch.tensor(labels,  dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.recordings[idx], dtype=torch.float32)
        return x, self.lengths[idx], self.labels[idx]


def collate_fn(batch):
    """
    Pad to the longest sequence in this batch only, then sort by
    descending length for pack_padded_sequence.
    """
    sequences, lengths, labels = zip(*batch)

    # Pad to batch-local maximum — not the global maximum
    sequences = torch.nn.utils.rnn.pad_sequence(
        sequences, batch_first=True, padding_value=0.0
    )
    lengths = torch.stack(lengths)
    labels  = torch.stack(labels)

    order     = lengths.argsort(descending=True)
    return sequences[order], lengths[order], labels[order]

In [9]:
class LSTMClassifier(nn.Module):

    def __init__(
        self,
        input_size: int,          # pass explicitly (was: default = num_channels)
        num_classes: int,         # pass explicitly (was: default = Num_IDs)
        lstm1_units: int = 128,
        lstm2_units: int = 64,
        dense_units: int = 64,
        recurrent_drop: float = 0.0,
        dense_drop: float = 0.0,
        bidirectional: bool = False,
    ):
        # NOTE: input_size and num_classes are now required arguments.
        # Previously they defaulted to the globals `num_channels` / `Num_IDs`,
        # which are bound when this cell runs. Changing the signal/object
        # selection and re-running the data cell without re-running this one
        # silently produced a model with the wrong input width or class count.
        super().__init__()
        self.bidirectional = bidirectional
        D = 2 if bidirectional else 1  # 2 if Bi-LSTM is selected.

        self.lstm1 = nn.LSTM(
            input_size = input_size,
            hidden_size = lstm1_units,
            num_layers = 1,
            batch_first = True,
            bidirectional = bidirectional,
        )
        self.drop1 = nn.Dropout(p=recurrent_drop)

        self.lstm2 = nn.LSTM(
            input_size = lstm1_units * D,  # input from layer 1 output
            hidden_size = lstm2_units,
            num_layers = 1,
            batch_first = True,
            bidirectional = bidirectional,
        )
        self.drop2 = nn.Dropout(p=recurrent_drop)

        self.dense_drop = nn.Dropout(p=dense_drop)
        self.fc1 = nn.Linear(lstm2_units * D, dense_units)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(dense_units, num_classes)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        packed1 = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=True)
        out1, _ = self.lstm1(packed1)
        out1, _ = nn.utils.rnn.pad_packed_sequence(out1, batch_first=True)
        out1 = self.drop1(out1)
        packed2 = pack_padded_sequence(out1, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (hidden_state, _) = self.lstm2(packed2) # hidden_state : (num_directions, batch, hidden_size)

        if self.bidirectional:
            final = torch.cat([hidden_state[0], hidden_state[1]], dim=-1)
        else:
            final = hidden_state[0]

        final = self.drop2(final)

        out = self.dense_drop(final)
        out = self.relu(self.fc1(out))
        out = self.fc2(out)
        return out

In [10]:
def run_epoch(model, loader, criterion, optimiser=None, train=True):
    """One full pass over a DataLoader. Returns (mean_loss, accuracy)."""
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for sequences, lengths, labels in loader:
            sequences = sequences.to(DEVICE)
            lengths   = lengths.to(DEVICE)
            labels    = labels.to(DEVICE)

            logits = model(sequences, lengths)
            loss   = criterion(logits, labels)

            if train:
                optimiser.zero_grad()
                loss.backward()
                # Gradient clipping — stabilises training on long sequences
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimiser.step()

            total_loss += loss.item() * labels.size(0)
            preds       = logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

    return total_loss / total, correct / total

def evaluate(model, loader, criterion):
    """Full evaluation pass. Returns loss, accuracy, predictions, true labels."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for sequences, lengths, labels in loader:
            sequences = sequences.to(DEVICE)
            lengths   = lengths.to(DEVICE)
            labels    = labels.to(DEVICE)

            logits = model(sequences, lengths)
            loss   = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            preds       = logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)

## Training

Preprocessing note: the per-channel mean and standard deviation used to
normalise the sequences are computed from the **training recordings only**
(all of their timesteps concatenated) and applied unchanged to the validation
and test recordings. Feeding the raw channels unscaled left them on wildly
different magnitudes, which pushes the LSTM gates toward saturation and also
made this model's preprocessing inconsistent with the feature-based
classifiers.

In [ ]:
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
random_state = 42
MAX_EPOCHS = 200
MODEL_PATH = "./best_lstm_model.pth"

# --- Fix #4: seed everything that affects the model, not just the split -----
# random_state alone only seeded train_test_split; weight initialisation,
# dropout masks and batch shuffling were left unseeded, so repeated runs of
# this cell gave different test accuracies.
np.random.seed(random_state)
torch.manual_seed(random_state)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_state)

# --- Regularisation (fix #3: these used to default to 0.0, i.e. none) ------
RECURRENT_DROP = 0.0    # dropout between the two LSTM layers
DENSE_DROP     = 0.3    # dropout before the dense head
BIDIRECTIONAL  = False

#Stratified 60 / 20 / 20 split
print("\n[2/5] Splitting data (60 / 20 / 20) ...")
indices = np.arange(len(labels))

#60% train, 40% temp — stratified to keep class balance in every split
idx_train, idx_temp, _, labels_temp = train_test_split(indices, labels, test_size=0.40, stratify=labels,random_state=random_state)

#50% of temp → val (20% overall), 50% → test (20% overall)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, stratify=labels_temp, random_state=random_state)

print("Train: {} samples".format(len(idx_train)))
print("Val: {} samples".format(len(idx_val)))
print("Test: {} samples".format(len(idx_test)))

# ===========================================================================
# PER-CHANNEL NORMALISATION - parameters estimated on the TRAINING split only
# ---------------------------------------------------------------------------
# The raw channels are different physical quantities whose magnitudes differ by
# orders of magnitude. Feeding them unscaled drives the LSTM gates towards
# saturation for the large channels while the small ones contribute almost
# nothing, and it makes this model's preprocessing inconsistent with the
# feature-based classifiers (which are standardised).
#
# Statistics are computed by concatenating all timesteps of the TRAINING
# recordings, so every timestep is weighted equally regardless of how long a
# given recording is. Validation and test recordings are transformed with those
# same frozen parameters - no statistic from them influences training.
#
# Side benefit: after this, padding_value=0.0 coincides with the training mean.
# (Packing already makes the padded region invisible to the LSTM, so this is
#  tidiness rather than a correctness requirement.)
# ===========================================================================
print("\n[2b/5] Normalising channels (training statistics only) ...")

train_stack = np.concatenate([recordings[i] for i in idx_train], axis=0)   # (sum_T, C)
chan_mu = train_stack.mean(axis=0)
chan_sd = train_stack.std(axis=0) + 1e-8
del train_stack

recordings_norm = [((rec - chan_mu) / chan_sd).astype(np.float32) for rec in recordings]

print("Per-channel training mean: {}".format(np.array2string(chan_mu, precision=3)))
print("Per-channel training std : {}".format(np.array2string(chan_sd, precision=3)))

_tr_chk = np.concatenate([recordings_norm[i] for i in idx_train], axis=0)
_te_chk = np.concatenate([recordings_norm[i] for i in idx_test], axis=0)
print("Check - normalised train mean ~0: {:+.2e}, std ~1: {:.4f}".format(_tr_chk.mean(), _tr_chk.std()))
print("Check - test mean is NOT forced to 0: {:+.4f} (expected, no leakage)".format(_te_chk.mean()))
del _tr_chk, _te_chk

#Datasets & DataLoaders
print("\n[3/5] Building datasets and loaders ...")

def make_loader(idx, shuffle):
    subset_recordings = [recordings_norm[i] for i in idx]   # normalised inputs
    subset_lengths    = lengths[idx]
    subset_labels     = labels[idx]

    ds = TimeSeriesDataset(subset_recordings, subset_lengths, subset_labels)
    return DataLoader(
        ds,
        batch_size = 128,
        shuffle    = shuffle,
        collate_fn = collate_fn,
        num_workers = 0,
        pin_memory  = (DEVICE.type == "cuda"),
    )

train_loader = make_loader(idx_train, shuffle=True)
val_loader = make_loader(idx_val, shuffle=False)
test_loader = make_loader(idx_test, shuffle=False)

#Model, loss, optimiser, scheduler
print("\n[4/5] Building model ...")
model = LSTMClassifier(
    input_size     = num_channels,   # passed explicitly (fix #5)
    num_classes    = Num_IDs,
    recurrent_drop = RECURRENT_DROP,
    dense_drop     = DENSE_DROP,
    bidirectional  = BIDIRECTIONAL,
).to(DEVICE)
print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("\nTrainable parameters: {}".format(n_params))

criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
#Halve LR when val loss stalls for 5 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode="min", factor=0.5, patience=5)

#Training loop with early stopping & best-weight checkpointing
print("\n[5/5] Training ...")
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss = float("inf")
epochs_no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimiser, train=True)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print("Epoch {}/{}  |  Training Loss: {:.4f};  Accuracy: {:.4f}  |  Validation Loss {:.4f};  Accuracy {:.4f}".format(epoch, MAX_EPOCHS, train_loss, train_acc, val_loss, val_acc))

    # Checkpoint best weights (chosen by validation loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print("✓ Best model saved  (val_loss={:.4f})".format(best_val_loss))
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= 10:
            print("\nEarly stopping triggered after {} epochs (10 epochs without improvement).".format(epoch))
            break

#Load best weights and evaluate on test set
print("\nLoading best weights from '{}' ...".format(MODEL_PATH))
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))

test_loss, test_acc, preds, true_labels = evaluate(model, test_loader, criterion)
print("\nTest Loss: {:.4f}; Accuracy: {:.4f}".format(test_loss, test_acc))
print("Classification report:")
# Fix #1: pin `labels` to the full class range and use real object names.
# Without labels=..., sklearn infers the classes present in y_true/y_pred; if a
# class is missing from the 20% test split this raised
#   ValueError: Number of classes, N, does not match size of target_names, M
print(classification_report(
    true_labels, preds,
    labels        = np.arange(Num_IDs),
    target_names  = class_names,
    zero_division = 0,
))


[2/5] Splitting data (60 / 20 / 20) ...
Train: 919 samples
Val: 306 samples
Test: 307 samples

[2b/5] Normalising channels (training statistics only) ...
Per-channel training mean: [25.247  0.987 32.302 32.162  8.519  0.122  0.129]
Per-channel training std : [11.824 15.784 19.435  6.913 10.136  0.022  0.029]
Check - normalised train mean ~0: -1.25e-08, std ~1: 1.0000
Check - test mean is NOT forced to 0: +0.0019 (expected, no leakage)

[3/5] Building datasets and loaders ...

[4/5] Building model ...
LSTMClassifier(
  (lstm1): LSTM(7, 128, batch_first=True)
  (drop1): Dropout(p=0.0, inplace=False)
  (lstm2): LSTM(128, 64, batch_first=True)
  (drop2): Dropout(p=0.0, inplace=False)
  (dense_drop): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=64, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=9, bias=True)
)

Trainable parameters: 124553

[5/5] Training ...


In [ ]:
epochs_ran = len(history["train_loss"])
x = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(x, history["train_loss"], label="Train")
axes[0].plot(x, history["val_loss"],   label="Val")
axes[0].set_title("Loss per epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend()

# Accuracy curve
axes[1].plot(x, history["train_acc"], label="Train")
axes[1].plot(x, history["val_acc"],   label="Val")
axes[1].set_title("Accuracy per epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

# Confusion matrix
cm = confusion_matrix(true_labels, preds, labels=np.arange(Num_IDs))
sns.heatmap(
    cm, ax=axes[2], fmt="d", cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
axes[2].set_title("Confusion Matrix — Test Set")
axes[2].set_xlabel("Predicted class")
axes[2].set_ylabel("True class")

plt.tight_layout()
#plt.savefig("training_results.png", dpi=150)
#print("\nPlots saved to training_results.png")
plt.show()
